In [1]:
import os
import sys
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler

# 1. Locate the physical directory of this notebook
notebook_path = Path(os.getcwd())

# 2. Walk up the directory tree until we find your data warehouse ROOT
# This loop handles nested subdirectories automatically
current_dir = notebook_path
while current_dir.name != "data_warehouse" and current_dir.parent != current_dir:
    current_dir = current_dir.parent

# Fallback: If the root folder name isn't exactly 'data_warehouse', assume it is the parent of your scripts folder
if current_dir.name != "data_warehouse":
    current_dir = notebook_path.parent

# 3. Append the absolute ROOT pathway to Python's structural import path
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

# 4. Enforce strict database path configuration matching your system layout
db_dir = current_dir / "databases"
sba_db_path = db_dir / "sba_7a_analysis.db"
bls_db_path = db_dir / "bls_laus_macro.db"
spatial_db_path = db_dir / "spatial_crosswalk.db"

# 5. Verify the import works flawlessly now
try:
    from utils.geography_daemon import GeographyDaemon
    print("=========================================================================")
    print("💎 SUCCESS: GeographyDaemon imported flawlessly from your ROOT engine!  ")
    print("=========================================================================")
    print(f"Data Warehouse ROOT verified at: {current_dir}")
    print(f"Spatial Database verified at:    {spatial_db_path.exists()}")
except ImportError as e:
    print(f"❌ Structural Import Error: {e}")
    print(f"Currently attempted lookup path was: {current_dir}")


💎 SUCCESS: GeographyDaemon imported flawlessly from your ROOT engine!  
Data Warehouse ROOT verified at: /Users/bonwier/PythonProjects/data_warehouse
Spatial Database verified at:    True


In [2]:

from utils.geography_daemon import GeographyDaemon  # Enabled by Cell 1's path injection


# =====================================================================
# 1. SETUP ENVIRONMENT & HIGH-SPEED CO-TEMPORAL BULK CACHE
# =====================================================================
sba_db_path = db_dir / "sba_7a_analysis.db"
spatial_db_path = db_dir / "spatial_crosswalk.db"

ag_naics_map = {
    '3118': 'Processing: Artisanal Bakeries',
    '3119': 'Processing: Specialty Foods',
    '3116': 'Processing: Meat & Slaughtering',
    '3114': 'Processing: Fruit/Veg Preserving',
    '3115': 'Processing: Dairy Manufacturing',
    '1114': 'Production: Controlled-Env Ag (CEA)',
    '1119': 'Production: Diversified Market Gardens',
    '1112': 'Production: Vegetable & Melon Farming',
    '1113': 'Production: Fruit & Tree Nut Farming',
    '4244': 'Pipelines: Food Hub Wholesalers',
    '1151': 'Pipelines: Crop Production Support',
    '1152': 'Pipelines: Animal Support/Veterinary',
    '1121': 'Pipelines: Boutique Livestock/Grazing'
}

# Generate SQL string filter tuple for the broad index match
sql_naics_tuple = tuple(ag_naics_map.keys())

# Load the 5,480-row crosswalk into memory once
crosswalk_conn = sqlite3.connect(spatial_db_path)
df_cw_cache = pd.read_sql_query("SELECT source_naics, target_naics FROM dim_naics_crosswalk WHERE mapping_type='NAICS_REVISION';", crosswalk_conn)
crosswalk_conn.close()

# Clean strings to build an exact dictionary lookup
df_cw_cache['source_naics'] = df_cw_cache['source_naics'].astype(str).str.strip()
df_cw_cache['target_naics'] = df_cw_cache['target_naics'].astype(str).str.strip()
naics_lookup_dict = dict(zip(df_cw_cache['source_naics'], df_cw_cache['target_naics']))

# =====================================================================
# 2. HIGH-SPEED SQL FILTER EXTRACTION (THE FIX)
# =====================================================================
sba_conn = sqlite3.connect(sba_db_path)

# SPEED BOOST: Filter down to your specific 13 targets DIRECTLY in SQLite. 
# This completely prevents loading a million unrelated rows into Pandas.
query_all_ag = f"""
SELECT locationid, approvaldate, paidinfulldate, chargeoffdate, isdefaulted, 
       terminmonths, grossapproval, naicscode, projectstate
FROM model_cohort_2003_present
WHERE SUBSTR(CAST(naicscode AS TEXT), 1, 4) IN {sql_naics_tuple};
"""
df_raw_ag = pd.read_sql_query(query_all_ag, sba_conn)
sba_conn.close()

print(f"[STAGE 1] Loaded {len(df_raw_ag)} target agricultural loan records from database.")

# =====================================================================
# 3. VECTORIZED DATE SORTING & TIMELINE TRACKING
# =====================================================================
df_raw_ag['approval_dt'] = pd.to_datetime(df_raw_ag['approvaldate'], errors='coerce')
df_raw_ag['paid_dt'] = pd.to_datetime(df_raw_ag['paidinfulldate'], errors='coerce')
df_raw_ag['chg_dt'] = pd.to_datetime(df_raw_ag['chargeoffdate'], errors='coerce')
df_raw_ag = df_raw_ag.dropna(subset=['approval_dt']).copy()
df_raw_ag['start_year'] = df_raw_ag['approval_dt'].dt.year.astype(int)

# Highly optimized vector calculation to replace row-by-row apply loops
snapshot_dt = pd.to_datetime('2026-06-20')
days_to_chg = (df_raw_ag['chg_dt'] - df_raw_ag['approval_dt']).dt.days
days_to_paid = (df_raw_ag['paid_dt'] - df_raw_ag['approval_dt']).dt.days
days_to_snapshot = (snapshot_dt - df_raw_ag['approval_dt']).dt.days

# Choose the active endpoint chronologically based on your data status logic
calculated_days = np.select(
    [df_raw_ag['chg_dt'].notna(), df_raw_ag['paid_dt'].notna()],
    [days_to_chg, days_to_paid],
    default=days_to_snapshot
)
df_raw_ag['duration_years'] = np.maximum(0.1, calculated_days / 365.25)
df_raw_ag['default_event'] = df_raw_ag['isdefaulted'].fillna(0).astype(int)

# =====================================================================
# 4. INSTANT IN-MEMORY DICTIONARY NAICS CROSSWALK
# =====================================================================
def fast_crosswalk(raw_naics):
    if pd.isna(raw_naics) or str(raw_naics).strip() == "":
        return None
    # Intercept float representation strings cleanly
    naics_str = str(raw_naics).strip().split('.')[0]
    
    # Trace code forward through changes up to 5 times natively in Python
    for _ in range(5):
        if naics_str in naics_lookup_dict:
            naics_str = naics_lookup_dict[naics_str]
        else:
            break
    return naics_str

# Apply crosswalk lookup across our small, filtered dataset
df_raw_ag['resolved_naics'] = df_raw_ag['naicscode'].apply(fast_crosswalk)
df_raw_ag['naics_4digit'] = df_raw_ag['resolved_naics'].astype(str).str.strip().str[:4]
df_raw_ag['industry_label'] = df_raw_ag['naics_4digit'].map(ag_naics_map)

# =====================================================================
# 5. STRUCTURE COMPOSITE TIERS & SCALE VALUES
# =====================================================================
df_raw_ag['is_short_term'] = (df_raw_ag['terminmonths'] <= 36).astype(int)
df_raw_ag['composite_strata_key'] = df_raw_ag['naics_4digit'] + "_" + df_raw_ag['is_short_term'].astype(str)

# Filter for the valid matches and apply Z-score normalization
df_portfolio = df_raw_ag.dropna(subset=['industry_label']).copy()
scaler = StandardScaler()
df_portfolio[['grossapproval']] = scaler.fit_transform(df_portfolio[['grossapproval']])

print(f"\n[SUCCESS] Connected Portfolio Engine completed processing.")
print(f"  • Total valid loan rows unlocked via Crosswalk: {len(df_portfolio)}")
print(f"  • Total distinct active pricing staircases: {df_portfolio['composite_strata_key'].nunique()}")



[STAGE 1] Loaded 26293 target agricultural loan records from database.

[SUCCESS] Connected Portfolio Engine completed processing.
  • Total valid loan rows unlocked via Crosswalk: 26293
  • Total distinct active pricing staircases: 26


In [3]:
# Model Fitting

# Initialize and execute the multi-sector stratified survival model
cph_portfolio = CoxPHFitter()
cph_portfolio.fit(
    df_portfolio[['duration_years', 'default_event', 'grossapproval', 'composite_strata_key']],
    duration_col='duration_years',
    event_col='default_event',
    strata=['composite_strata_key']
)

print("3. Stratified Multi-Sector Optimization Complete.")

3. Stratified Multi-Sector Optimization Complete.


In [4]:
# Extract the baseline cumulative hazard matrices out of the trained fitter
portfolio_cum_hazards = cph_portfolio.baseline_cumulative_hazard_
portfolio_summary_rows = []

# Map targets back to clean tracking intervals
target_years = [1.0, 2.0, 3.0, 4.0, 5.0]

# Evaluate each industry family across both short-term and long-term asset variations
for naics_4, ind_name in ag_naics_map.items():
    for capital_flag, cap_name in [(1, 'Short-Term Line'), (0, 'Long-Term Debt')]:
        strata_lookup_key = f"{naics_4}_{capital_flag}"
        
        # Verify if the database slice populated this specific staircase variant
        if strata_lookup_key in portfolio_cum_hazards.columns:
            strata_curve = portfolio_cum_hazards[strata_lookup_key]
            
            # Extract cumulative hazard positions via nearest-index queries
            idx_3 = strata_curve.index.get_indexer([3.0], method='nearest')[0]
            idx_5 = strata_curve.index.get_indexer([5.0], method='nearest')[0]
            
            cum_h_3 = strata_curve.iloc[idx_3]
            cum_h_5 = strata_curve.iloc[idx_5]
            
            # Translate absolute cumulative hazard bounds into true cumulative PD percentages
            pd_3yr = (1 - np.exp(-cum_h_3)) * 100
            pd_5yr = (1 - np.exp(-cum_h_5)) * 100
            
            portfolio_summary_rows.append({
                'Industry Family': ind_name,
                'Capital Structure Tier': cap_name,
                '3-Year Cumulative PD (%)': round(pd_3yr, 2),
                '5-Year Cumulative PD (%)': round(pd_5yr, 2)
            })

# Compile and sort the findings from highest risk velocity to lowest
df_portfolio_exposure_matrix = pd.DataFrame(portfolio_summary_rows)
df_portfolio_exposure_matrix = df_portfolio_exposure_matrix.sort_values(by=['5-Year Cumulative PD (%)'], ascending=False)

# FIXED: Wrapped in triple-quotes to prevent the unterminated string syntax error
print("""=========================================================================")
               PERI-URBAN PORTFOLIO CREDIT RISK EXPOSURE MATRIX          
=========================================================================""")
df_portfolio_exposure_matrix.head(25) # Renders the top 25 risk profiles in your ag pool


=========================================================================")
               PERI-URBAN PORTFOLIO CREDIT RISK EXPOSURE MATRIX          


,Industry Family,Capital Structure Tier,3-Year Cumulative PD (%),5-Year Cumulative PD (%)
22,Pipelines: Animal Support/Veterinary,Short-Term Line,2.62,44.18
14,Production: Vegetable & Melon Farming,Short-Term Line,19.11,36.88
18,Pipelines: Food Hub Wholesalers,Short-Term Line,5.45,27.57
10,Production: Controlled-Env Ag (CEA),Short-Term Line,4.09,26.09
0,Processing: Artisanal Bakeries,Short-Term Line,4.82,25.22
2,Processing: Specialty Foods,Short-Term Line,3.26,23.59
24,Pipelines: Boutique Livestock/Grazing,Short-Term Line,1.84,20.46
6,Processing: Fruit/Veg Preserving,Short-Term Line,1.65,15.43
4,Processing: Meat & Slaughtering,Short-Term Line,2.34,15.22
12,Production: Diversified Market Gardens,Short-Term Line,7.85,12.03


In [5]:
# 1. Pull the 5-year true risk profiles calculated directly from your Stratified Cox Model above
# We build a dictionary from your actual df_portfolio_exposure_matrix data
strata_risk_lookup = df_portfolio_exposure_matrix.set_index(['Industry Family', 'Capital Structure Tier'])['5-Year Cumulative PD (%)'].to_dict()

# 2. Configure the Master Pool Parameters
TOTAL_POOL_CORPUS = 10_000_000.00
NUM_SIMULATED_LOANS = 100
INITIAL_SLICE_VALUE = 100_000.00  # Each loan gets a $100k guarantee slice from the pool (Bridges a $500k loan)
ANNUAL_STEP_DOWN_RATE = 0.15     # Guarantee value drops by 15% each year as loan principal amortizes

# 3. Build a Mock Portfolio Mix (80% Short-Term Working Capital / 20% Long-Term Assets)
np.random.seed(42)
portfolio_loans = []

# Available staircases from your matrix
short_tier_keys = df_portfolio_exposure_matrix[df_portfolio_exposure_matrix['Capital Structure Tier'] == 'Short-Term Line']['Industry Family'].tolist()
long_tier_keys = df_portfolio_exposure_matrix[df_portfolio_exposure_matrix['Capital Structure Tier'] == 'Long-Term Debt']['Industry Family'].tolist()

for i in range(NUM_SIMULATED_LOANS):
    is_short = np.random.choice([True, False], p=[0.80, 0.20])
    if is_short:
        ind = np.random.choice(short_tier_keys)
        cap_tier = 'Short-Term Line'
    else:
        ind = np.random.choice(long_tier_keys)
        cap_tier = 'Long-Term Debt'
        
    portfolio_loans.append({'loan_id': f"Loan_{i:03d}", 'industry': ind, 'tier': cap_tier})

# 4. Run the 5-Year Financial Operational Simulation Loop
pool_balance = TOTAL_POOL_CORPUS
premium_reserve = 0.0
simulation_records = []

for year_idx in range(1, 6):
    year_label = f"Year {year_idx}"
    
    # Calculate this year's guarantee slice value after step-down depreciation
    current_slice_value = INITIAL_SLICE_VALUE * ((1 - ANNUAL_STEP_DOWN_RATE) ** (year_idx - 1))
    total_active_exposure = len(portfolio_loans) * current_slice_value
    
    annual_premium_collected = 0.0
    annual_losses_incurred = 0.0
    active_loans_before_defaults = len(portfolio_loans)
    
    surviving_loans = []
    
    # Process each active loan through the annual survival gates
    for loan in portfolio_loans:
        ind = loan['industry']
        tier = loan['tier']
        
        # Pull true historical conditional risk profile from your Cox model findings
        # If missing, fallback to the overall average risk
        total_5yr_risk = strata_risk_lookup.get((ind, tier), 5.0) / 100.0
        annual_conditional_pd = total_5yr_risk / 5.0  # Linearized annual default slice
        
        # Pricing Rules:
        if tier == 'Long-Term Debt':
            # Your custom rule: Fixed 3% annual premium
            premium_rate = 0.03 
        else:
            # Short-Term rule: Slightly over-priced and front-loaded to manage the infant mortality bump
            premium_rate = max(0.045, annual_conditional_pd * 1.25)
            
        # Collect Premium based on this year's stepping-down exposure value
        loan_premium = current_slice_value * premium_rate
        annual_premium_collected += loan_premium
        
        # Roll the dice: Simulate default event based on true survival probabilities
        if np.random.rand() < annual_conditional_pd:
            # Default triggers! Pool pays out 100% of this year's active slice value to the CDFI
            annual_losses_incurred += current_slice_value
        else:
            # Loan survives cleanly to buy option renewal next year
            surviving_loans.append(loan)
            
    # Update portfolio volume tracking
    portfolio_loans = surviving_loans
    premium_reserve += annual_premium_collected
    premium_reserve -= annual_losses_incurred
    
    simulation_records.append({
        'Timeline': year_label,
        'Active Guarantees': active_loans_before_defaults,
        'Exposure Per Loan ($)': round(current_slice_value, 2),
        'Total Pool Exposure ($)': round(total_active_exposure, 2),
        'Premiums Collected ($)': round(annual_premium_collected, 2),
        'Default Payouts ($)': round(annual_losses_incurred, 2),
        'Net Premium Reserve Cushion ($)': round(premium_reserve, 2)
    })

df_simulation_dashboard = pd.DataFrame(simulation_records)
print("""=========================================================================")
          EVERGREEN CORPUS: 5-YEAR STEP-DOWN OPTION RUNWAY SIMULATOR         
=========================================================================""")
df_simulation_dashboard

=========================================================================")
          EVERGREEN CORPUS: 5-YEAR STEP-DOWN OPTION RUNWAY SIMULATOR         


,Timeline,Active Guarantees,Exposure Per Loan ($),Total Pool Exposure ($),Premiums Collected ($),Default Payouts ($),Net Premium Reserve Cushion ($)
0,Year 1,100,100000.00,10000000.0,531347.50,100000.00,431347.50
1,Year 2,99,85000.00,8415000.0,442257.12,340000.00,533604.62
2,Year 3,95,72250.00,6863750.0,350045.83,72250.00,811400.46
3,Year 4,94,61412.50,5772775.0,290755.95,122825.00,979331.40
4,Year 5,92,52200.62,4802457.5,242444.50,52200.62,1169575.27
